# Run a full load locally with checkpoint-based processing

This notebook runs the v5 pipeline against a local payload file directly,
bypassing Flask entirely. No HTTP timeouts, no client polling.

**Key properties:**
- Calls `process_load_with_checkpoints` directly (same code path as the
  batch script).
- Writes per-plan checkpoints to Azure blob storage if configured;
  otherwise falls back to local disk.
- Resumable: if you interrupt (Ctrl+C or kernel restart), re-run this
  notebook. Completed plans are skipped.
- Live progress via `clear_output` so the cell output stays manageable
  even on a 2+ hour run.

**Requirements:**
- `build_benefits.py`, `checkpoint_runner.py` must be importable
  (in the same folder as this notebook, or on the Python path).
- Azure OpenAI credentials in env vars: `AZURE_OPENAI_ENDPOINT`,
  `AZURE_OPENAI_API_KEY`.
- `BLOB_CONNECTION_STRING` optional (skips blob, uses local disk if absent).


## 0. Config - paths and IDs

In [ ]:
import json
import os
import sys
import time
import threading
from pathlib import Path
from datetime import datetime, timezone

# Ensure imports work when notebook is run from any folder
sys.path.insert(0, str(Path.cwd()))

# ----- Local file locations -----
# Path to the payload JSON file. Either the raw {pbp: [...]} dict or a bare
# list of PBP rows is accepted.
PAYLOAD_PATH = Path(r'C:\\Users\\i40212888\\Downloads\\210-20260511T174006Z_payload.json')

# Path to prompts. Either point to local files (PROMPT_DIR) OR set
# BLOB_PROMPTS_CONTAINER env var to load from blob.
PROMPT_DIR   = Path(r'C:\\Users\\i40212888\\Downloads\\prompts')   # ignored if blob loader works

# Load ID - used in checkpoint blob names. Re-using the same ID resumes a run.
LOAD_ID = '210'

# ----- Output -----
# Where to write the combined final output if running fully local (no blob)
LOCAL_OUTPUT_PATH = Path.cwd() / f'output_benefits_{LOAD_ID}.json'

# Where to keep checkpoints if blob isn't available
LOCAL_CHECKPOINT_DIR = Path.cwd() / 'local_checkpoints'

print(f'Payload:           {PAYLOAD_PATH}')
print(f'Load ID:           {LOAD_ID}')
print(f'BLOB available:    {bool(os.getenv("BLOB_CONNECTION_STRING"))}')
print(f'Local checkpoint:  {LOCAL_CHECKPOINT_DIR}')
print(f'Local output:      {LOCAL_OUTPUT_PATH}')


## 1. Load the payload and prompts locally

If your prompts live in blob, prefer blob (set `BLOB_CONNECTION_STRING` and
`BLOB_PROMPTS_CONTAINER`). Otherwise this loads them from local disk.

In [ ]:
# Payload
payload_raw = json.loads(PAYLOAD_PATH.read_text(encoding='utf-8'))
pbp_rows = payload_raw['pbp'] if isinstance(payload_raw, dict) and 'pbp' in payload_raw else payload_raw
print(f'Loaded {len(pbp_rows):,} PBP rows from {PAYLOAD_PATH.name}')

# Prompts - try blob first, then local
prompts = None
if os.getenv('BLOB_CONNECTION_STRING'):
    try:
        from azure.storage.blob import BlobServiceClient
        svc = BlobServiceClient.from_connection_string(os.environ['BLOB_CONNECTION_STRING'])
        cc = svc.get_container_client(os.getenv('BLOB_PROMPTS_CONTAINER', 'prompts'))
        prompts = {
            'system_prompt':     cc.download_blob('system_prompt.txt').readall().decode('utf-8'),
            'few_shot_examples': cc.download_blob('few_shot_examples.txt').readall().decode('utf-8'),
            'human_template':    cc.download_blob('human_template.txt').readall().decode('utf-8'),
        }
        print(f'Loaded prompts from BLOB container "{cc.container_name}"')
    except Exception as e:
        print(f'Blob prompt load failed ({e}), falling back to local')

if prompts is None:
    prompts = {
        'system_prompt':     (PROMPT_DIR / 'system_prompt.txt').read_text(encoding='utf-8'),
        'few_shot_examples': (PROMPT_DIR / 'few_shot_examples.txt').read_text(encoding='utf-8'),
        'human_template':    (PROMPT_DIR / 'human_template.txt').read_text(encoding='utf-8'),
    }
    print(f'Loaded prompts from LOCAL dir {PROMPT_DIR}')

for key, val in prompts.items():
    print(f'  {key:<22} {len(val):>6,} chars')


## 2. Verify the build_benefits version

Stops immediately if the code on disk doesn't match what you expect.
Prevents the 'I ran the test against stale code' time-waster.

In [ ]:
from build_benefits import __BUILD_VERSION__
print(f'build_benefits version: {__BUILD_VERSION__}')

EXPECTED_VERSION_PREFIX = 'v5'   # accept v5 and v5.x variants
if EXPECTED_VERSION_PREFIX not in __BUILD_VERSION__:
    raise SystemExit(
        f'WARNING - build_benefits is {__BUILD_VERSION__}, expected to contain "{EXPECTED_VERSION_PREFIX}". '
        f'Check the file on disk is the v5 architecture (sweeper + retry + checkpoints).'
    )
print('OK - v5 architecture present')


## 3. Decide: blob checkpoints or local disk?

If `BLOB_CONNECTION_STRING` is set, `checkpoint_runner` will write to blob
(production-like). Otherwise the notebook falls back to a local-disk
implementation - same logic, files on your laptop instead of in blob.

In [ ]:
USE_BLOB = bool(os.getenv('BLOB_CONNECTION_STRING'))

if USE_BLOB:
    from checkpoint_runner import process_load_with_checkpoints, read_status
    print('Using BLOB checkpoints (production code path)')
else:
    # Local-disk fallback. Mirrors process_load_with_checkpoints' behaviour but
    # writes JSON files to LOCAL_CHECKPOINT_DIR.
    from build_benefits import group_rows_by_plan, run_one_plan_processing
    import re
    import traceback

    LOCAL_CHECKPOINT_DIR.mkdir(exist_ok=True)
    print(f'Using LOCAL checkpoints at {LOCAL_CHECKPOINT_DIR}')

    def _safe_fn(name):
        return re.sub(r'[^A-Za-z0-9._-]+', '_', name).strip('_')

    def _local_checkpoint_path(load_id, plan_filename):
        return LOCAL_CHECKPOINT_DIR / f'checkpoint_{load_id}_{_safe_fn(plan_filename)}.json'

    def _local_status_path(load_id):
        return LOCAL_CHECKPOINT_DIR / f'status_{load_id}.json'

    def _utc_now():
        return datetime.now(timezone.utc).isoformat().replace('+00:00', 'Z')

    def process_load_local(load_id, pbp_rows, prompts):
        '''Local-disk version of process_load_with_checkpoints.'''
        plan_groups = group_rows_by_plan(pbp_rows)
        plan_filenames = list(plan_groups.keys())

        # Init/resume status
        status_path = _local_status_path(load_id)
        if status_path.exists():
            status = json.loads(status_path.read_text())
            for fn in plan_filenames:
                if fn not in status['plans']:
                    status['plans'][fn] = {'status': 'pending'}
            status['status'] = 'processing'
            print(f'Resuming - {status.get("n_plans_done", 0)} plan(s) already done')
        else:
            status = {
                'status':        'processing',
                'load_id':       load_id,
                'started_at':    _utc_now(),
                'updated_at':    _utc_now(),
                'completed_at':  None,
                'n_plans_total': len(plan_filenames),
                'n_plans_done':  0,
                'n_plans_failed': 0,
                'plans':         {fn: {'status': 'pending'} for fn in plan_filenames},
            }

        def save_status():
            status['updated_at'] = _utc_now()
            status_path.write_text(json.dumps(status, indent=2))

        save_status()

        for i, fn in enumerate(plan_filenames, 1):
            cp = _local_checkpoint_path(load_id, fn)
            if cp.exists():
                if status['plans'][fn].get('status') != 'done':
                    rows = json.loads(cp.read_text())
                    status['plans'][fn] = {'status': 'done', 'rows': len(rows)}
                    status['n_plans_done'] = sum(1 for p in status['plans'].values() if p.get('status') == 'done')
                    save_status()
                print(f'[{i:>3}/{len(plan_filenames)}] SKIP  {fn}  (checkpoint exists)')
                continue

            plan_rows = plan_groups[fn]
            print(f'[{i:>3}/{len(plan_filenames)}] PROCESS  {fn}  ({len(plan_rows):,} rows)')
            status['plans'][fn] = {'status': 'processing', 'started_at': _utc_now()}
            save_status()

            t0 = time.monotonic()
            try:
                out = run_one_plan_processing(plan_rows, prompts)
                elapsed = time.monotonic() - t0
                cp.write_text(json.dumps(out))
                status['plans'][fn] = {
                    'status': 'done', 'rows': len(out), 'elapsed_s': round(elapsed, 1),
                }
                status['n_plans_done'] = sum(1 for p in status['plans'].values() if p.get('status') == 'done')
                save_status()
                print(f'           DONE in {elapsed:.1f}s - {len(out)} rows')
            except Exception as e:
                elapsed = time.monotonic() - t0
                status['plans'][fn] = {
                    'status': 'failed', 'error': str(e), 'elapsed_s': round(elapsed, 1),
                }
                status['n_plans_failed'] = sum(1 for p in status['plans'].values() if p.get('status') == 'failed')
                save_status()
                print(f'           FAILED after {elapsed:.1f}s: {e}')
                traceback.print_exc()

        # Combine
        combined = []
        missing = []
        for fn in plan_filenames:
            cp = _local_checkpoint_path(load_id, fn)
            if cp.exists():
                combined.extend(json.loads(cp.read_text()))
            else:
                missing.append(fn)

        LOCAL_OUTPUT_PATH.write_text(json.dumps(combined, indent=2))
        status['status'] = 'success' if not missing and status['n_plans_failed'] == 0 else 'partial'
        status['completed_at'] = _utc_now()
        status['combined_rows_count'] = len(combined)
        status['output_path'] = str(LOCAL_OUTPUT_PATH)
        save_status()

        return {
            'load_id':            load_id,
            'n_plans_total':      len(plan_filenames),
            'n_plans_done':       status['n_plans_done'],
            'n_plans_failed':     status['n_plans_failed'],
            'n_plans_pending':    len(missing) - status['n_plans_failed'],
            'combined_rows_count': len(combined),
            'output_path':        str(LOCAL_OUTPUT_PATH),
            'status':             status['status'],
        }


## 4. Live progress monitor (runs in a background thread)

While the main processing cell runs, this thread refreshes a single
output line every 30s with current plan counts. Won't crash the kernel
because it uses `clear_output(wait=True)`.

In [ ]:
from IPython.display import clear_output, display

_MONITOR_STOP = threading.Event()

def _read_current_status():
    '''Read the current run status from blob or local disk.'''
    if USE_BLOB:
        try:
            return read_status(LOAD_ID)
        except Exception:
            return None
    else:
        sp = _local_status_path(LOAD_ID)
        if sp.exists():
            return json.loads(sp.read_text())
        return None

def _monitor_loop():
    while not _MONITOR_STOP.is_set():
        s = _read_current_status()
        if s:
            n_done = s.get('n_plans_done', 0)
            n_total = s.get('n_plans_total', '?')
            n_failed = s.get('n_plans_failed', 0)
            clear_output(wait=True)
            print(f'[{datetime.now().strftime("%H:%M:%S")}] '
                  f'status={s.get("status")}  '
                  f'plans={n_done}/{n_total}  '
                  f'failed={n_failed}  '
                  f'updated={s.get("updated_at")}')
        _MONITOR_STOP.wait(30)

def start_monitor():
    _MONITOR_STOP.clear()
    t = threading.Thread(target=_monitor_loop, daemon=True)
    t.start()
    return t

def stop_monitor():
    _MONITOR_STOP.set()

print('Monitor helpers ready. Call start_monitor() before the run cell,')
print('stop_monitor() after. Or just run the next cell which does both.')


## 5. Run the full load

This cell runs to completion (or you interrupt it). Re-running this cell
after an interruption picks up from the last completed plan.

**Expect this cell to take a long time.** A 67-plan load with the sweeper
on runs ~2.5h. You can interrupt and resume freely - no work is lost.

In [ ]:
# Start the monitor thread for live progress updates
monitor = start_monitor()

try:
    t_start = time.monotonic()
    if USE_BLOB:
        summary = process_load_with_checkpoints(
            load_id=LOAD_ID,
            pbp_rows=pbp_rows,
            prompts=prompts,
            force_reprocess=False,
            max_plans_this_run=None,
        )
    else:
        summary = process_load_local(LOAD_ID, pbp_rows, prompts)
    total_elapsed = time.monotonic() - t_start
finally:
    stop_monitor()

clear_output(wait=False)
print('=' * 70)
print(f'Run complete in {total_elapsed:.1f}s ({total_elapsed/60:.1f} min)')
print('=' * 70)
print(f'Status:              {summary["status"]}')
print(f'Plans total:         {summary["n_plans_total"]}')
print(f'Plans done:          {summary["n_plans_done"]}')
print(f'Plans failed:        {summary["n_plans_failed"]}')
print(f'Plans pending:       {summary["n_plans_pending"]}')
print(f'Combined rows out:   {summary["combined_rows_count"]:,}')
if USE_BLOB:
    print(f'Output blob:         outbound/{summary["output_blob"]}')
else:
    print(f'Output file:         {summary["output_path"]}')


## 6. Inspect the combined output

Quick sanity checks on what came out. Run this after section 5 finishes
successfully (or after a partial completion - the combined file is written
even when not all plans are done).

In [ ]:
import pandas as pd

# Read the combined output
if USE_BLOB:
    from azure.storage.blob import BlobServiceClient
    svc = BlobServiceClient.from_connection_string(os.environ['BLOB_CONNECTION_STRING'])
    cc = svc.get_container_client(os.getenv('BLOB_OUTBOUND_CONTAINER', 'outbound'))
    out_data = json.loads(cc.download_blob(f'output_benefits_{LOAD_ID}.json').readall().decode('utf-8'))
else:
    out_data = json.loads(LOCAL_OUTPUT_PATH.read_text(encoding='utf-8'))

print(f'Total rows: {len(out_data):,}\n')

df = pd.DataFrame(out_data)
if not df.empty:
    print(f'Distinct plans:        {df["planid"].nunique()}')
    print(f'Distinct benefit IDs:  {df["benefitid"].nunique()}')
    print()
    print('Rows per plan (top 15):')
    rows_per_plan = df.groupby('planid').size().sort_values(ascending=False)
    print(rows_per_plan.head(15).to_string())
    print()
    print('Rows per benefit ID (all):')
    rows_per_benefit = df.groupby('benefitid').size().sort_values(ascending=False)
    print(rows_per_benefit.to_string())


## 7. Optional - retry failed plans only

If section 5 ended with `status: partial` and some plans failed, this cell
force-reprocesses ONLY the failed ones.

In [ ]:
# Read the most recent status
status = _read_current_status()

if not status:
    print('No status to inspect')
else:
    failed_plans = [name for name, info in status.get('plans', {}).items()
                    if info.get('status') == 'failed']
    print(f'Failed plans: {len(failed_plans)}')
    for fn in failed_plans:
        print(f'  - {fn}: {status["plans"][fn].get("error", "")[:120]}')

    # To retry: delete the checkpoint(s) for failed plans, then re-run cell 5.
    # Uncomment the lines below to do that automatically.
    # if not USE_BLOB:
    #     for fn in failed_plans:
    #         cp = _local_checkpoint_path(LOAD_ID, fn)
    #         if cp.exists():
    #             cp.unlink()
    #             print(f'   Deleted local checkpoint for {fn}')
    #     print(f'\nRe-run section 5 to retry these plans.')
